# 🧪 Phase 06 — Robustness & Ablation (cross-symbol, cross-layer, cross-horizon)

This notebook does **not** retrain the full model roster -- Phase 05 (`Phase05_Unified_L1_v4`,
`Copy_of_Phase05_Unified_L2_v4`, `Copy_of_Phase05_Unified_L3_v4`) already did that, and already
persisted every leaderboard, every TEST-set prediction, every MI/SHAP ranking table, and every
ablation/significance test to a **stable, symbol/layer/horizon-keyed `Artifacts/` tree**
(Section 2 below). This notebook's job, per the project review, is to be the
**journal-validation notebook**: pull all of that together into the cross-symbol / cross-layer /
cross-horizon robustness picture that no single Phase-05 run can show on its own, and add the
handful of genuinely new checks the review calls for that Phase 05 could not produce by itself
(Feature-Count Sensitivity, Walk-Forward Stability).

**Sections** (matching the review's recommended `Phase06_Robustness_and_Ablation.ipynb` outline):

1. Load frozen Phase-03 features
2. Load frozen feature-selection results
3. Purged Walk-Forward CV
4. Information-Set Ablation (H2) — aggregated across symbols/layers/horizons
5. Regime Ablation (H3) — aggregated across symbols/layers/horizons
6. Feature-Count Sensitivity — **new**: 20 / 40 / 60 / 80 / 100 features, fixed model
7. Regime-Threshold Sensitivity — aggregated (computed per-run inside Phase 05 L1)
8. Walk-Forward Stability — **new**: fixed model, purged-CV fold-to-fold variance
9. Predictive-Superiority Tests (McNemar / Diebold-Mariano / Bootstrap CI) — aggregated
10. Regime-Conditional Performance — aggregated + a new cross-layer (L1→L2/L3) join
11. Horizon-Conditional Performance — aggregated
12. Cross-Index Heterogeneity — aggregated (NIFTY50 vs BANKNIFTY vs NIFTYNEXT50)
13. Economic Significance — aggregated
14. Robustness Summary Tables — one consolidated scorecard

Sections 4/5/7/9/10/11/12/13 are **pure aggregation** of what Phase 05 already saved -- run Phase
05's three notebooks across whichever symbols/horizons you want covered first, then run this
notebook once to build the combined picture. Sections 6 and 8 **do** retrain (one small, fixed,
default-hyperparameter model -- never the full roster) because those two checks genuinely cannot
be produced from Phase 05's saved artifacts alone.


## 1 · Install & Imports

In [ ]:
!pip install -q lightgbm shap scikit-learn pandas numpy scipy pyarrow fastparquet tqdm


In [ ]:
import warnings, os, glob, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, norm as scipy_norm

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, matthews_corrcoef, roc_auc_score, log_loss, mean_squared_error,
    mean_absolute_error, r2_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import TimeSeriesSplit

try:
    import lightgbm as lgb
except ImportError:
    get_ipython().system('pip install -q lightgbm')
    import lightgbm as lgb

try:
    import shap
except ImportError:
    get_ipython().system('pip install -q shap')
    import shap

from IPython.display import display

print('✅ Imports ready.')


## 2 · Config

In [ ]:
# ── ⭐ Grid this notebook aggregates / sweeps over ⭐ ────────────────────────────────────────
# Must match Phase 05's own SYMBOL / HORIZON choices exactly -- this notebook reads the SAME
# Artifacts/ tree those three notebooks wrote to, keyed by (symbol, layer, horizon).
ALL_SYMBOLS = ['NIFTY50', 'BANKNIFTY', 'NIFTYNEXT50']
ALL_HORIZONS_FOR_STATS = [1, 3, 5, 10, 21]

# The single symbol/horizon this run's NEW computations (Sections 6/8 in Phase06, Q4/Q5 in
# Phase07) retrain against -- exactly the "two things you change" pattern from Phase 05. The
# pure-aggregation sections above/below read the FULL grid (ALL_SYMBOLS × ALL_HORIZONS_FOR_STATS)
# regardless of this setting.
SYMBOL  = 'NIFTYNEXT50'
HORIZON = 3

RANDOM_STATE = 42

# ── Paths -- IDENTICAL to Phase05_Unified_L1_v4 / L2_v4 / L3_v4's own Section 2 ─────────────
GDRIVE_BASE = '/content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107'
DATA_ROOT   = os.path.join(GDRIVE_BASE, 'Data')
FEAT_DIR    = os.path.join(DATA_ROOT, 'features_v6')
FS_DIR      = os.path.join(DATA_ROOT, 'feature_selection_v6')

ARTIFACT_ROOT     = os.path.join(GDRIVE_BASE, 'Artifacts')
MODEL_SAVE_DIR    = os.path.join(ARTIFACT_ROOT, 'models')
PRED_SAVE_DIR     = os.path.join(ARTIFACT_ROOT, 'predictions')
FEATIMP_SAVE_DIR  = os.path.join(ARTIFACT_ROOT, 'feature_importance')
RESULTS_SAVE_DIR  = os.path.join(ARTIFACT_ROOT, 'results')
STATS_SAVE_DIR    = os.path.join(ARTIFACT_ROOT, 'stats')

# ── Layer registry -- every sub-layer target Phase 05 trains, its task, and its target column ──
LAYER_TASK = {
    'L1_1_TRI': 'classification', 'L1_1_DIR': 'classification',
    'L1_2_RVOL': 'regression', 'L1_2_MAXRET': 'regression', 'L1_2_MAXDRAW': 'regression',
    'L1_3_RET': 'regression', 'L1_3_PCT': 'regression',
}
LAYER_TARGET_COL = {
    'L1_1_TRI': 'target_tri_h', 'L1_1_DIR': 'target_dir_h',
    'L1_2_RVOL': 'target_fwd_rvol_h', 'L1_2_MAXRET': 'target_max_ret_h', 'L1_2_MAXDRAW': 'target_max_draw_h',
    'L1_3_RET': 'target_fwd_ret_h', 'L1_3_PCT': 'target_fwd_pct_h',
}
ALL_LAYERS = list(LAYER_TASK.keys())
CLASSIFICATION_LAYERS = [l for l, t in LAYER_TASK.items() if t == 'classification']
REGRESSION_LAYERS     = [l for l, t in LAYER_TASK.items() if t == 'regression']

# ── L1.1 (classification) class encoding -- IDENTICAL to Phase05_Unified_L1_v4 ──────────────
CLASS_ORDER      = [-1, 0, 1]
REGIME_LABEL_MAP = {-1: 'Bear', 0: 'Sideways', 1: 'Bull'}
ENCODE_MAP        = {-1: 0, 0: 1, 1: 2}
DECODE_MAP        = {v: k for k, v in ENCODE_MAP.items()}

# ── Feature-Selection Config -- IDENTICAL to Phase05_Unified_L1_v4 Section 2.1 (needed so that
#    Section 6's feature-count sweep reruns the EXACT SAME selection pipeline, just at different
#    top_k, rather than a different one that would confound "feature count" with "selection
#    method") ──────────────────────────────────────────────────────────────────────────────
CORR_THRESHOLD  = 0.75
MISSING_MAX     = 0.30
FS_RANDOM_STATE = RANDOM_STATE
FINAL_SELECTION_BASIS = 'shap'
MI_RANK_WEIGHT, SHAP_RANK_WEIGHT = 0.50, 0.50
SHAP_ENABLED, SHAP_MAX_SAMPLES, SHAP_N_ESTIMATORS = True, 2000, 300
META_COLS = {'date', 'symbol', 'open', 'high', 'low', 'close', 'volume', 'adj_close'}

DOMAIN_GROUPS = {
    'core': ['1. Raw Price & Return', '2. Candle-Body & Microstructure', '3. Momentum Indicators',
              '4. Trend Indicators', '5. Volatility Indicators', '6. Volume Indicators',
              '7. Rolling Statistical', '8. Lag Features', '9. Calendar & Regime'],
    'auxiliary': ['10. External Derived Features', '12. Slope & Acceleration Features',
                  '13. Interaction Features', '14. Advanced Regime, Volatility & Percentile'],
    'excluded': ['0. Cross-Market & Relative Strength', '11. Intraday Features'],
}
GROUP_RATIOS = {'core': 0.80, 'auxiliary': 0.20, 'excluded': 0.00}

# ── Regime-label economics (must match Phase-03's target_tri_{h}d construction) ─────────────
REGIME_THRESHOLD = 0.005

# ── Purged Walk-Forward CV (Section 3) ──────────────────────────────────────────────────────
USE_PURGED_CV = True
PURGE_DAYS    = HORIZON
EMBARGO_DAYS  = HORIZON
WF_SPLITS     = 5   # more folds than Phase 05's stacking OOF (3) -- this notebook's CV is used
                     # for fold-level STABILITY diagnostics (Sections 8 / Q5), which wants more,
                     # smaller folds than an OOF-feature generator does.

# ── This notebook's ONE fixed, default-hyperparameter model, used for every retraining-based
#    check below (Sections 6/8, Q4/Q5) -- same "carefully selected model, not more algorithms"
#    principle as Phase 05's own Section 9b ablation toolkit. ────────────────────────────────
ROBUSTNESS_MODEL_NAME = 'LightGBM_Robustness'
FEATURE_COUNT_GRID    = [20, 40, 60, 80, 100]

for _d in (MODEL_SAVE_DIR, PRED_SAVE_DIR, FEATIMP_SAVE_DIR, RESULTS_SAVE_DIR, STATS_SAVE_DIR):
    os.makedirs(_d, exist_ok=True)

print(f'✅ Config loaded. Default sweep target: SYMBOL={SYMBOL}  HORIZON={HORIZON}d')
print(f'   ALL_SYMBOLS={ALL_SYMBOLS}')
print(f'   ALL_HORIZONS_FOR_STATS={ALL_HORIZONS_FOR_STATS}')
print(f'   ALL_LAYERS={ALL_LAYERS}')
print(f'   ARTIFACT_ROOT = {ARTIFACT_ROOT}')


### 2.1 · Domain / Pattern Rules & Information-Set Buckets — IDENTICAL to Phase 05

`DOMAIN_PATTERN_RULES` (fine-grained, 15 domains) is copied verbatim from Phase 05's Section 2.2
-- used here for the feature-count sweep's own `select_features_train_only()` call and for
`mi_shap_*.csv`'s `domain` column. `INFO_SET_PATTERN_RULES` (coarse, 6 buckets: Market / Macro /
GPR / Sentiment / Sustainability / Regime, plus "Cross-market" added here for the
`0. Cross-Market & Relative Strength` domain the review's Q2 explicitly lists) is copied verbatim
from Phase 05 L2/L3's Section 9b ablation toolkit -- used for Phase 07's domain-level SHAP
aggregation.


In [ ]:
# ── Feature-name -> fine-grained domain (IDENTICAL to Phase05_Unified_L1_v4 Section 2.2) ──────
DOMAIN_PATTERN_RULES = [
    ('peer_', '0. Cross-Market & Relative Strength'), ('xmk_', '0. Cross-Market & Relative Strength'),
    ('rel_mom_', '0. Cross-Market & Relative Strength'), ('rel_atr_', '0. Cross-Market & Relative Strength'),
    ('rel_rsi_', '0. Cross-Market & Relative Strength'), ('rel_drawdown_', '0. Cross-Market & Relative Strength'),
    ('rel_breakout_', '0. Cross-Market & Relative Strength'), ('outperform_', '0. Cross-Market & Relative Strength'),

    ('price_', '1. Raw Price & Return'), ('log_close', '1. Raw Price & Return'),
    ('log_volume', '1. Raw Price & Return'), ('ret_', '1. Raw Price & Return'),
    ('log_ret', '1. Raw Price & Return'), ('gap', '1. Raw Price & Return'),
    ('range_', '1. Raw Price & Return'), ('dist_from_', '1. Raw Price & Return'),
    ('drawdown_', '1. Raw Price & Return'), ('distance_52w', '1. Raw Price & Return'),
    ('illiquidity_', '1. Raw Price & Return'),

    ('candle_', '2. Candle-Body & Microstructure'), ('is_doji', '2. Candle-Body & Microstructure'),
    ('is_hammer', '2. Candle-Body & Microstructure'), ('is_shooting_star', '2. Candle-Body & Microstructure'),
    ('close_in_range', '2. Candle-Body & Microstructure'),

    ('mom_', '3. Momentum Indicators'), ('trend_', '4. Trend Indicators'), ('vola_', '5. Volatility Indicators'),

    ('vol_of_vol_', '14. Advanced Regime, Volatility & Percentile'),
    ('vol_overnight_20', '14. Advanced Regime, Volatility & Percentile'),
    ('vol_intraday_20', '14. Advanced Regime, Volatility & Percentile'),
    ('overnight_intraday_vol_ratio', '14. Advanced Regime, Volatility & Percentile'),
    ('regime_p_', '14. Advanced Regime, Volatility & Percentile'), ('pctile_', '14. Advanced Regime, Volatility & Percentile'),
    ('z_volume_', '14. Advanced Regime, Volatility & Percentile'), ('z_atr_', '14. Advanced Regime, Volatility & Percentile'),
    ('z_mom_', '14. Advanced Regime, Volatility & Percentile'), ('z_vix_', '14. Advanced Regime, Volatility & Percentile'),
    ('days_since_', '14. Advanced Regime, Volatility & Percentile'), ('bull_streak', '14. Advanced Regime, Volatility & Percentile'),
    ('bear_streak', '14. Advanced Regime, Volatility & Percentile'), ('vol_spike_streak', '14. Advanced Regime, Volatility & Percentile'),
    ('consec_', '14. Advanced Regime, Volatility & Percentile'), ('atr_slope', '14. Advanced Regime, Volatility & Percentile'),
    ('atr_accel', '14. Advanced Regime, Volatility & Percentile'), ('atr_increasing', '14. Advanced Regime, Volatility & Percentile'),
    ('realized_quarticity', '14. Advanced Regime, Volatility & Percentile'), ('bipower_variation', '14. Advanced Regime, Volatility & Percentile'),
    ('jump_component', '14. Advanced Regime, Volatility & Percentile'),

    ('vol_', '6. Volume Indicators'), ('stat_', '7. Rolling Statistical'), ('lag_', '8. Lag Features'),
    ('cal_', '9. Calendar & Regime'), ('regime_', '9. Calendar & Regime'),
    ('intra_', '11. Intraday Features'),
    ('india_vix_slope', '12. Slope & Acceleration Features'), ('india_vix_accel', '12. Slope & Acceleration Features'),
    ('gold_trend_slope', '12. Slope & Acceleration Features'), ('yield_spread_slope', '12. Slope & Acceleration Features'),
    ('sentiment_slope', '12. Slope & Acceleration Features'),
    ('ix3_', '13. Interaction Features'), ('ix_', '13. Interaction Features'),

    ('beta_', '10. External Derived Features'), ('corr_', '10. External Derived Features'),
    ('rs_', '10. External Derived Features'), ('gpr_', '10. External Derived Features'),
    ('sustain_gpr', '10. External Derived Features'), ('sent_', '10. External Derived Features'),
    ('net_sentiment', '10. External Derived Features'), ('av_sent', '10. External Derived Features'),
    ('gdelt1_', '10. External Derived Features'), ('gdelt2_', '10. External Derived Features'),
    ('riskcat_', '10. External Derived Features'), ('count_financial', '10. External Derived Features'),
    ('count_political', '10. External Derived Features'), ('count_climate', '10. External Derived Features'),
    ('event_', '10. External Derived Features'), ('high_event', '10. External Derived Features'),
    ('fred_', '10. External Derived Features'), ('rate_', '10. External Derived Features'),
    ('fx_', '10. External Derived Features'), ('inf_', '10. External Derived Features'),
    ('mon_', '10. External Derived Features'), ('act_', '10. External Derived Features'),
    ('lead_', '10. External Derived Features'), ('trade_', '10. External Derived Features'),
    ('yield_curve_inverted', '10. External Derived Features'), ('cpu_', '10. External Derived Features'),
    ('climate_', '10. External Derived Features'), ('green_transition', '10. External Derived Features'),
    ('esg_', '10. External Derived Features'), ('sustain_', '10. External Derived Features'),
    ('india_vix', '10. External Derived Features'), ('usdinr', '10. External Derived Features'),
    ('gold_usd', '10. External Derived Features'), ('gold_safehaven', '10. External Derived Features'),
    ('market_stress', '10. External Derived Features'), ('ext_missing_source', '10. External Derived Features'),
    ('ext_sentiment_momentum', '10. External Derived Features'), ('ext_', '10. External Derived Features'),
]


def get_domain(col: str) -> str:
    for prefix, domain in DOMAIN_PATTERN_RULES:
        if col.startswith(prefix):
            return domain
    return 'Unmapped'


def domain_breakdown(feature_list):
    doms = pd.Series([get_domain(f) for f in feature_list], name='domain')
    out = doms.value_counts().rename_axis('domain').reset_index(name='n_features')
    return out.sort_values('n_features', ascending=False).reset_index(drop=True)


# ── Fine-grained domain -> coarse information-set bucket (IDENTICAL bucketing rule to Phase05
#    L2/L3's Section 9b, PLUS 'Cross-market' for the review's Q2 list) -- built from the SAME
#    prefixes so a feature maps to exactly one bucket regardless of which notebook asks. ─────
INFO_SET_PATTERN_RULES = [
    ('peer_', 'Cross-market'), ('xmk_', 'Cross-market'), ('rel_mom_', 'Cross-market'),
    ('rel_atr_', 'Cross-market'), ('rel_rsi_', 'Cross-market'), ('rel_drawdown_', 'Cross-market'),
    ('rel_breakout_', 'Cross-market'), ('outperform_', 'Cross-market'),
    ('regime_', 'Regime'),
    ('gpr_', 'GPR'), ('sustain_gpr', 'GPR'),
    ('sent_', 'Sentiment'), ('net_sentiment', 'Sentiment'), ('av_sent', 'Sentiment'),
    ('gdelt1_', 'Sentiment'), ('gdelt2_', 'Sentiment'), ('riskcat_', 'Sentiment'),
    ('count_financial', 'Sentiment'), ('count_political', 'Sentiment'), ('count_climate', 'Sentiment'),
    ('event_', 'Sentiment'), ('high_event', 'Sentiment'),
    ('esg_', 'Sustainability'), ('sustain_', 'Sustainability'), ('climate_', 'Sustainability'),
    ('green_transition', 'Sustainability'), ('cpu_', 'Sustainability'),
    ('fred_', 'Macro'), ('rate_', 'Macro'), ('fx_', 'Macro'), ('inf_', 'Macro'), ('mon_', 'Macro'),
    ('act_', 'Macro'), ('lead_', 'Macro'), ('trade_', 'Macro'), ('yield_curve_inverted', 'Macro'),
    ('yield_spread_slope', 'Macro'), ('usdinr', 'Macro'), ('gold_usd', 'Macro'),
    ('gold_trend_slope', 'Macro'), ('gold_safehaven', 'Macro'), ('india_vix', 'Macro'),
    ('market_stress', 'Macro'), ('ext_missing_source', 'Macro'), ('ext_sentiment_momentum', 'Macro'),
    ('ext_', 'Macro'),
]
INFO_SET_CUMULATIVE_ORDER = ['Market', 'Macro', 'GPR', 'Sentiment', 'Sustainability']


def get_info_set(col: str) -> str:
    for prefix, bucket in INFO_SET_PATTERN_RULES:
        if col.startswith(prefix):
            return bucket
    return 'Market'


print(f'✅ DOMAIN_PATTERN_RULES: {len(DOMAIN_PATTERN_RULES)} rules -> '
      f'{len(set(d for _, d in DOMAIN_PATTERN_RULES))} fine-grained domains.')
print(f'✅ INFO_SET_PATTERN_RULES: {len(INFO_SET_PATTERN_RULES)} rules -> coarse buckets '
      f'{sorted(set(b for _, b in INFO_SET_PATTERN_RULES) | {"Market"})}')


## 3 · Load Frozen Phase-03 Features + Frozen Phase-05 Feature-Selection Results

Per the review's outline, this notebook loads the SAME `{symbol}_features_{split}.parquet`
matrices Phase 05 reads (`load_raw_splits`, `clean_xy` — ported verbatim), plus whichever of Phase
05's ALREADY-SAVED, TRAIN-only-selected feature lists it finds under `FS_DIR` (written there by
Phase 05's own `save_selected_features()`). Nothing is re-selected unless a symbol/layer/horizon
combination has genuinely never been run in Phase 05 yet, in which case Section 6's feature-count
sweep runs `select_features_train_only()` itself (also ported verbatim) — the review's own
"feature-count sensitivity" ask is, after all, a request to re-run selection at different `top_k`.


In [ ]:
def load_raw_splits(symbol: str) -> dict:
    """IDENTICAL to Phase05_Unified_L1_v4's load_raw_splits(). Already contains every target_*
    column computed upstream in Phase-03 -- no target is ever recreated here."""
    sym_dir = os.path.join(FEAT_DIR, symbol)
    splits = {}
    for split in ['train', 'val', 'test']:
        path = os.path.join(sym_dir, f'{symbol}_features_{split}.parquet')
        if os.path.exists(path):
            splits[split] = pd.read_parquet(path)
        else:
            print(f'  ⚠️ Not found: {path}')
    return splits


def clean_xy(df: pd.DataFrame, feat_cols: list, target_col: str) -> pd.DataFrame:
    """IDENTICAL to Phase05_Unified_L1_v4's clean_xy()."""
    cols = feat_cols + [target_col]
    d = df[cols].replace([np.inf, -np.inf], np.nan).dropna(subset=[target_col]).copy()
    d[feat_cols] = d[feat_cols].fillna(d[feat_cols].median()).fillna(0)
    return d


def get_candidate_features(df: pd.DataFrame) -> list:
    """IDENTICAL to Phase05_Unified_L1_v4's get_candidate_features()."""
    return [c for c in df.columns if c not in META_COLS and not c.startswith('target_')
            and pd.api.types.is_numeric_dtype(df[c])]


def load_frozen_selection(symbol: str, layer: str, horizon: int):
    """Reads Phase 05's ALREADY-SAVED selection for this (symbol, layer, horizon), written by
    save_selected_features() under FS_DIR/{symbol}/{layer}/{symbol}_selection_{layer}_{h}d.txt
    (+ the matching _ranking_{layer}_{h}d.csv). Returns (feats, ranking_df) or ([], None) if
    Phase 05 has not been run for this combination yet."""
    sel_dir = os.path.join(FS_DIR, symbol, layer)
    feat_path = os.path.join(sel_dir, f'{symbol}_selection_{layer}_{horizon}d.txt')
    rank_path = os.path.join(sel_dir, f'{symbol}_ranking_{layer}_{horizon}d.csv')
    if not os.path.exists(feat_path):
        return [], None
    with open(feat_path) as f:
        feats = [line.strip() for line in f if line.strip()]
    ranking = pd.read_csv(rank_path) if os.path.exists(rank_path) else None
    return feats, ranking


def build_xy_from_frozen(symbol, layer, horizon, split, raw_splits=None):
    """Loads `split` for `symbol` (or reuses `raw_splits` if already loaded) and slices it down
    to Phase 05's frozen selected features for (symbol, layer, horizon). Returns (X, y, index) or
    None if either the split or the frozen selection is unavailable."""
    raw_splits = raw_splits if raw_splits is not None else load_raw_splits(symbol)
    df = raw_splits.get(split)
    feats, _ = load_frozen_selection(symbol, layer, horizon)
    target_col = LAYER_TARGET_COL[layer]
    if df is None or not feats or target_col not in df.columns:
        return None
    d = clean_xy(df, feats, target_col)
    return d[feats], d[target_col], d.index


print('load_raw_splits(), clean_xy(), get_candidate_features(), load_frozen_selection(), '
      'build_xy_from_frozen() ready.')


### 3.1 · Train-Only Selection Pipeline (ported verbatim) — used only by Section 6's feature-count sweep


In [ ]:
def _rank_to_score(rank_series):
    n = rank_series.notna().sum()
    if n <= 1:
        return pd.Series(1.0, index=rank_series.index, dtype=float)
    return 1.0 - (rank_series - 1) / (n - 1)


def _compute_shap_importance(X_imp, y, features, task, random_state):
    """IDENTICAL to Phase05_Unified_L1_v4's _compute_shap_importance(): train-only SHAP via an RF
    surrogate, on up to SHAP_MAX_SAMPLES rows. Reused by both the feature-count sweep (Section 6)
    and every SHAP computation in Phase 07."""
    if not SHAP_ENABLED or not features:
        return pd.Series(dtype=float, name='shap_importance')
    try:
        X_sel = X_imp[features].copy()
        max_samples = min(SHAP_MAX_SAMPLES, len(X_sel))
        if max_samples <= 0:
            return pd.Series(dtype=float, name='shap_importance')
        X_shap = X_sel.sample(n=max_samples, random_state=random_state) if len(X_sel) > max_samples else X_sel
        y_shap = y.loc[X_shap.index]
        if task == 'classification':
            surrogate = RandomForestClassifier(n_estimators=SHAP_N_ESTIMATORS, random_state=random_state,
                                                n_jobs=-1, class_weight='balanced')
        else:
            surrogate = RandomForestRegressor(n_estimators=SHAP_N_ESTIMATORS, random_state=random_state, n_jobs=-1)
        surrogate.fit(X_shap, y_shap)
        explainer = shap.TreeExplainer(surrogate)
        shap_values = explainer.shap_values(X_shap)
        if isinstance(shap_values, list):
            arr = np.stack([np.asarray(v) for v in shap_values], axis=0)
            mean_abs = np.mean(np.abs(arr), axis=(0, 1))
        else:
            arr = np.asarray(shap_values)
            mean_abs = np.mean(np.abs(arr), axis=0) if arr.ndim == 2 else np.mean(np.abs(arr), axis=(0, 2))
        return pd.Series(mean_abs, index=features, name='shap_importance').sort_values(ascending=False)
    except Exception as exc:
        print(f'  ⚠️ SHAP importance failed; returning empty. Reason: {type(exc).__name__}: {exc}')
        return pd.Series(dtype=float, name='shap_importance')


def select_features_train_only(train_df, candidate_features, target_col, task='classification',
                                top_k=40, corr_threshold=CORR_THRESHOLD, missing_max=MISSING_MAX,
                                random_state=FS_RANDOM_STATE, domain_groups=DOMAIN_GROUPS,
                                group_ratios=GROUP_RATIOS, verbose=True):
    """IDENTICAL pipeline to Phase05_Unified_L1_v4's select_features_train_only(): missingness ->
    variance -> correlation -> MI -> SHAP -> Unmapped removal -> domain-group cap -> final top_k.
    The ONLY reason this notebook ever calls it is Section 6's feature-count sweep -- everywhere
    else uses Phase 05's already-frozen selection (load_frozen_selection above)."""
    X = train_df[candidate_features].apply(pd.to_numeric, errors='coerce')
    y = train_df[target_col]
    valid = y.notna()
    X, y = X[valid], y[valid]

    missing_pct = X.isna().mean()
    keep = missing_pct[missing_pct <= missing_max].index.tolist()
    X = X[keep]

    X_imp_all = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X), columns=X.columns, index=X.index)
    std = X_imp_all.std()
    variance_keep = std[std > 1e-12].index.tolist()
    X_imp = X_imp_all[variance_keep]
    if X_imp.shape[1] == 0:
        return [], pd.DataFrame(columns=['feature', 'mi', 'mi_rank', 'shap_importance', 'shap_rank', 'domain', 'group'])

    mi_fn = mutual_info_classif if task == 'classification' else mutual_info_regression
    y_mi = y.astype(int) if task == 'classification' else y.astype(float)
    mi_pre = mi_fn(X_imp, y_mi, random_state=random_state)
    pre_order = pd.Series(mi_pre, index=X_imp.columns).sort_values(ascending=False).index.tolist()

    keep_corr = []
    corr = X_imp[pre_order].corr().abs()
    for feature in pre_order:
        if not keep_corr or float(corr.loc[feature, keep_corr].max()) < corr_threshold:
            keep_corr.append(feature)
    X_corr = X_imp[keep_corr]

    mi_final = mi_fn(X_corr, y_mi, random_state=random_state)
    ranking = pd.DataFrame({'feature': X_corr.columns, 'mi': mi_final}).sort_values('mi', ascending=False).reset_index(drop=True)
    ranking['mi_rank'] = ranking.index + 1
    ranking['domain'] = ranking['feature'].map(get_domain)

    shap_imp = _compute_shap_importance(X_corr, y, ranking['feature'].tolist(), task, random_state)
    ranking['shap_importance'] = ranking['feature'].map(shap_imp)
    ranking['shap_rank'] = ranking['feature'].map({f: i + 1 for i, f in enumerate(shap_imp.index)}) if len(shap_imp) else np.nan

    mapped_mask = ranking['domain'].ne('Unmapped')
    ranking_mapped = ranking.loc[mapped_mask].copy()
    if ranking_mapped.empty:
        return [], ranking

    domain_groups = domain_groups or {}
    group_ratios = group_ratios or {}
    domain_to_group = {d: g for g, doms in domain_groups.items() for d in doms}
    ranking_mapped['group'] = ranking_mapped['domain'].map(lambda d: domain_to_group.get(d, 'core'))
    ranking_mapped['mi_score'] = _rank_to_score(ranking_mapped['mi_rank'])
    ranking_mapped['shap_score'] = _rank_to_score(ranking_mapped['shap_rank'])

    if FINAL_SELECTION_BASIS == 'mi':
        ranking_mapped['final_score'] = ranking_mapped['mi_score']
        ranking_mapped = ranking_mapped.sort_values(['final_score', 'mi', 'feature'], ascending=[False, False, True])
    elif FINAL_SELECTION_BASIS == 'shap':
        ranking_mapped['final_score'] = ranking_mapped['shap_score']
        ranking_mapped = ranking_mapped.sort_values(['final_score', 'shap_importance', 'feature'], ascending=[False, False, True], na_position='last')
    else:
        w_mi = MI_RANK_WEIGHT / (MI_RANK_WEIGHT + SHAP_RANK_WEIGHT)
        w_shap = SHAP_RANK_WEIGHT / (MI_RANK_WEIGHT + SHAP_RANK_WEIGHT)
        ranking_mapped['final_score'] = w_mi * ranking_mapped['mi_score'] + w_shap * ranking_mapped['shap_score'].fillna(0.0)
        ranking_mapped = ranking_mapped.sort_values(['final_score', 'mi', 'shap_importance', 'feature'], ascending=[False, False, False, True], na_position='last')

    excluded_groups = {g for g, r in group_ratios.items() if r == 0}
    group_caps = {g: round(top_k * r) for g, r in group_ratios.items() if g not in excluded_groups}
    eligible = ranking_mapped[~ranking_mapped['group'].isin(excluded_groups)].copy()

    selected, group_counts = [], {}
    for _, row in eligible.iterrows():
        if len(selected) >= top_k:
            break
        g = row['group']
        cap = group_caps.get(g)
        if cap is not None and group_counts.get(g, 0) >= cap:
            continue
        selected.append(row['feature'])
        group_counts[g] = group_counts.get(g, 0) + 1
    if len(selected) < min(top_k, len(eligible)):
        for _, row in eligible.iterrows():
            if len(selected) >= top_k:
                break
            if row['feature'] not in selected:
                selected.append(row['feature'])

    if verbose:
        print(f'  select_features_train_only: top_k={top_k} -> selected {len(selected)} features '
              f'from {len(eligible)} group-eligible candidates')
    return selected, ranking


print('_compute_shap_importance(), select_features_train_only() ready (ported verbatim from Phase 05).')


## 4 · Purged Walk-Forward CV (embargo) — IDENTICAL to Phase05_Unified_L1_v4 Section 7b

Used by Section 8 (Walk-Forward Stability) and Q5 (SHAP fold-stability) below to generate
leakage-safe, non-overlapping-at-the-boundary folds for the fixed `ROBUSTNESS_MODEL_NAME` model.


In [ ]:
class PurgedWalkForwardCV:
    """IDENTICAL to Phase05_Unified_L1_v4's PurgedWalkForwardCV -- expanding-window CV with a
    purge window before, and an embargo window after, each validation fold."""
    def __init__(self, n_splits=5, purge=0, embargo=0):
        self.n_splits = n_splits
        self.purge = max(int(purge), 0)
        self.embargo = max(int(embargo), 0)

    def split(self, X, y=None, groups=None):
        n = len(X)
        tscv = TimeSeriesSplit(n_splits=self.n_splits)
        for train_idx, val_idx in tscv.split(X):
            val_start, val_end = val_idx[0], val_idx[-1]
            purge_start = max(val_start - self.purge, 0)
            train_idx_purged = train_idx[train_idx < purge_start]
            embargo_end = min(val_end + self.embargo + 1, n)
            train_idx_final = train_idx_purged[(train_idx_purged < val_start) | (train_idx_purged >= embargo_end)]
            if len(train_idx_final) == 0 or len(val_idx) == 0:
                continue
            yield train_idx_final, val_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


def get_cv_splitter(n_splits=WF_SPLITS):
    if USE_PURGED_CV:
        return PurgedWalkForwardCV(n_splits=n_splits, purge=PURGE_DAYS, embargo=EMBARGO_DAYS)
    return TimeSeriesSplit(n_splits=n_splits)


print(f'PurgedWalkForwardCV / get_cv_splitter() ready. USE_PURGED_CV={USE_PURGED_CV} '
      f'n_splits={WF_SPLITS} purge={PURGE_DAYS}d embargo={EMBARGO_DAYS}d')


## 5 · Shared Evaluation Helpers + the ONE Fixed Robustness Model

`evaluate_classification()` / `evaluate_regression()` are ported verbatim from Phase 05 (so every
metric name and definition matches exactly). `build_lightgbm_fixed()` is the single,
default-hyperparameter model every retraining-based section below uses -- deliberately not tuned
per arm, so that any difference across feature counts / folds / regimes reflects the *condition*
being swept, not a hyperparameter search finding a different optimum each time.


In [ ]:
def evaluate_classification(y_true, y_pred, y_proba=None):
    y_true = pd.Series(y_true).astype(int).values
    y_pred = pd.Series(y_pred).astype(int).values
    row = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', labels=CLASS_ORDER, zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', labels=CLASS_ORDER, zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', labels=CLASS_ORDER, zero_division=0),
        'cohen_kappa': cohen_kappa_score(y_true, y_pred, labels=CLASS_ORDER),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'roc_auc_ovr': np.nan, 'log_loss': np.nan,
    }
    if y_proba is not None:
        try:
            y_true_bin = label_binarize(y_true, classes=CLASS_ORDER)
            row['roc_auc_ovr'] = roc_auc_score(y_true_bin, y_proba, average='macro', multi_class='ovr')
        except Exception:
            pass
        try:
            row['log_loss'] = log_loss(y_true, y_proba, labels=CLASS_ORDER)
        except Exception:
            pass
    return row


def evaluate_regression(y_true, y_pred, y_ref=None, dir_mode='median', n_features=None):
    y_true = np.asarray(y_true, dtype=float); y_pred = np.asarray(y_pred, dtype=float)
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) == 0:
        return {'rmse': np.nan, 'mae': np.nan, 'r2': np.nan, 'spearman_corr': np.nan,
                'directional_acc': np.nan, 'forecast_bias': np.nan}
    r2 = r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan
    row = {'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))), 'mae': float(mean_absolute_error(y_true, y_pred)),
           'r2': float(r2), 'spearman_corr': np.nan, 'directional_acc': np.nan,
           'forecast_bias': float(np.mean(y_pred - y_true))}
    try:
        corr, _ = spearmanr(y_true, y_pred)
        row['spearman_corr'] = float(corr)
    except Exception:
        pass
    if dir_mode == 'zero':
        row['directional_acc'] = float(np.mean(np.sign(y_true) == np.sign(y_pred)))
    else:
        ref = y_ref if y_ref is not None else np.median(y_true)
        row['directional_acc'] = float(((y_true >= ref).astype(int) == (y_pred >= ref).astype(int)).mean())
    return row


def evaluate(task, y_true, y_pred, y_proba=None, y_ref=None, dir_mode='median', n_features=None):
    return (evaluate_classification(y_true, y_pred, y_proba=y_proba) if task == 'classification'
            else evaluate_regression(y_true, y_pred, y_ref=y_ref, dir_mode=dir_mode, n_features=n_features))


def build_lightgbm_fixed(task):
    """The ONE model every Section-6/8/Q4/Q5 retraining call below uses -- default
    hyperparameters, identical shape to Phase05's own build_lightgbm()."""
    default = dict(n_estimators=300, num_leaves=31, learning_rate=0.05, subsample=0.8,
                    colsample_bytree=0.8, random_state=RANDOM_STATE, verbosity=-1)
    if task == 'classification':
        default['objective'] = 'multiclass' if len(CLASS_ORDER) > 2 else 'binary'
        if len(CLASS_ORDER) > 2:
            default['num_class'] = len(CLASS_ORDER)
        return lgb.LGBMClassifier(**default)
    default['objective'] = 'regression'
    return lgb.LGBMRegressor(**default)


print('evaluate(), build_lightgbm_fixed(task) ready.')


## 6 · Cross-Grid Artifact Loader

One function, `load_artifact_grid(subdir, filename)`, powers almost every aggregation section
below: it scans `Artifacts/{subdir}/{symbol}/{layer}/h{horizon}d/{filename}` across every
`(symbol, layer, horizon)` in `ALL_SYMBOLS × ALL_LAYERS × ALL_HORIZONS_FOR_STATS`, reads whatever
Phase 05 has already saved there, and concatenates it into one long-format DataFrame — tagged with
the `symbol`/`layer`/`horizon` it came from even if the file itself didn't already have those
columns. Missing combinations (a symbol/horizon/layer Phase 05 hasn't been run for yet) are
silently skipped, exactly like Phase 05's own `aggregate_*_across_horizons()` helpers, just
generalised to the full grid instead of one symbol/layer at a time.


In [ ]:
def load_artifact_grid(subdir, filename, symbols=ALL_SYMBOLS, layers=None,
                        horizons=ALL_HORIZONS_FOR_STATS, verbose=False):
    """subdir: one of 'results', 'stats', 'feature_importance', 'predictions'. filename: e.g.
    'leaderboard.csv', 'information_set_ablation.csv', 'diebold_mariano.csv', 'mi_shap_train.csv'.
    layers=None -> every layer in ALL_LAYERS. Returns one concatenated DataFrame (empty if
    nothing found), with symbol/layer/horizon columns added whenever the source file lacks them
    (most already carry these -- see Phase05's save functions -- but this makes the loader safe
    regardless)."""
    layers = layers or ALL_LAYERS
    root = {'results': RESULTS_SAVE_DIR, 'stats': STATS_SAVE_DIR,
            'feature_importance': FEATIMP_SAVE_DIR, 'predictions': PRED_SAVE_DIR}[subdir]
    frames = []
    n_found, n_missing = 0, 0
    for symbol in symbols:
        for layer in layers:
            for horizon in horizons:
                path = os.path.join(root, symbol, layer, f'h{horizon}d', filename)
                if not os.path.exists(path):
                    n_missing += 1
                    continue
                try:
                    df = pd.read_csv(path)
                except (pd.errors.EmptyDataError, pd.errors.ParserError):
                    n_missing += 1
                    continue
                if df.empty:
                    continue
                for col, val in [('symbol', symbol), ('layer', layer), ('horizon', horizon)]:
                    if col not in df.columns:
                        df.insert(0, col, val)
                frames.append(df)
                n_found += 1
    if verbose or n_found == 0:
        print(f'  load_artifact_grid({subdir}/*/{{layer}}/h*/{filename}): '
              f'{n_found} files found, {n_missing} combinations not yet run in Phase 05.')
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


print('load_artifact_grid(subdir, filename, symbols=..., layers=..., horizons=...) ready.')


## 7 · Information-Set Ablation (H2) — Aggregated Across Symbols / Layers / Horizons

Reads every `information_set_ablation.csv` Phase 05 (L2's Sections 11.22/11B.22/11C.22 and L3's
Sections 12.22/12B.22) has already saved, across the full `ALL_SYMBOLS × ALL_HORIZONS_FOR_STATS`
grid. One row per (symbol, layer, horizon, arm). The pivot below shows the mean incremental
`ΔRMSE` each information-set arm adds, averaged across symbols — the cross-symbol version of the
"does external information help" question H2 asks, rather than a single symbol's answer.


In [ ]:
info_set_grid = load_artifact_grid('stats', 'information_set_ablation.csv',
                                    layers=REGRESSION_LAYERS, verbose=True)
print(f'\nTotal H2 information-set ablation rows loaded: {len(info_set_grid)}')
if not info_set_grid.empty:
    display(info_set_grid.head(20))

    pivot_delta_rmse = (
        info_set_grid.dropna(subset=['delta_rmse_vs_prev_arm'])
        .groupby(['layer', 'arm'])['delta_rmse_vs_prev_arm']
        .agg(['mean', 'std', 'count'])
        .reset_index()
        .sort_values(['layer', 'arm'])
    )
    print('\nMean incremental ΔRMSE per arm, averaged across ALL symbols/horizons already run:')
    display(pivot_delta_rmse)

    out_dir = os.path.join(STATS_SAVE_DIR, '_phase06_summary')
    os.makedirs(out_dir, exist_ok=True)
    info_set_grid.to_csv(os.path.join(out_dir, 'information_set_ablation_full_grid.csv'), index=False)
    pivot_delta_rmse.to_csv(os.path.join(out_dir, 'information_set_ablation_summary.csv'), index=False)
    print(f'\n💾 Saved full grid + summary -> {out_dir}')
else:
    print('  ⚠️ Nothing found yet -- run Phase05 L2/L3 for at least one symbol/horizon first.')


### 7b · H1 Formal Experiment — Aggregated

Phase 05 L1 (Sections 13.2/13B.2) already builds the explicit Benchmark → Proposed Model →
Improvement → CI → Significance table for the ternary and binary regime-detection layers. This
aggregates `h1_experiment.csv` across the grid — the cross-symbol/cross-horizon answer to H1
itself, not just H2/H3's supporting ablations.


In [ ]:
h1_grid = load_artifact_grid('stats', 'h1_experiment.csv', layers=CLASSIFICATION_LAYERS, verbose=True)
print(f'\nTotal H1 formal-experiment rows loaded: {len(h1_grid)}')
if not h1_grid.empty:
    display(h1_grid.sort_values(['layer', 'symbol', 'horizon']))
    h1_summary = (
        h1_grid.assign(significant=lambda d: d['significant_at_0.05'].fillna(False).astype(bool))
        .groupby(['layer', 'benchmark'])
        .agg(n_combinations=('significant', 'size'), pct_significant=('significant', 'mean'),
             mean_improvement=('improvement', 'mean'))
        .reset_index()
    )
    print('\nH1 verdict by layer/benchmark, across every symbol/horizon already run:')
    display(h1_summary)

    out_dir = os.path.join(STATS_SAVE_DIR, '_phase06_summary')
    os.makedirs(out_dir, exist_ok=True)
    h1_grid.to_csv(os.path.join(out_dir, 'h1_experiment_full_grid.csv'), index=False)
    h1_summary.to_csv(os.path.join(out_dir, 'h1_experiment_summary.csv'), index=False)
    print(f'\n💾 Saved -> {out_dir}')
else:
    print('  ⚠️ Nothing found yet -- run Phase05 L1 for at least one symbol/horizon first.')


## 8 · Regime-Information Ablation (H3) — Aggregated Across Symbols / Layers / Horizons

Reads every `regime_ablation.csv` Phase 05 L2/L3 has already saved. Positive `delta_rmse` means
regime information helped that (symbol, layer, horizon); the summary below reports what fraction
of already-run combinations show a significant (`dm_p_value < 0.05`) improvement, per layer —
the cross-symbol H3 verdict, not just one symbol's.


In [ ]:
regime_abl_grid = load_artifact_grid('stats', 'regime_ablation.csv', layers=REGRESSION_LAYERS, verbose=True)
print(f'\nTotal H3 regime-ablation rows loaded: {len(regime_abl_grid)}')
if not regime_abl_grid.empty:
    display(regime_abl_grid)

    summary_rows = []
    for layer, g in regime_abl_grid.groupby('layer'):
        sig = g['dm_significant_at_0.05'].fillna(False).astype(bool)
        summary_rows.append({
            'layer': layer, 'n_combinations': len(g),
            'n_regime_helps_significantly': int((sig & (g['delta_rmse'] > 0)).sum()),
            'n_regime_hurts_significantly': int((sig & (g['delta_rmse'] < 0)).sum()),
            'n_not_significant': int((~sig).sum()),
            'mean_delta_rmse': float(g['delta_rmse'].mean()),
        })
    regime_abl_summary = pd.DataFrame(summary_rows)
    print('\nH3 verdict by layer, across every symbol/horizon already run:')
    display(regime_abl_summary)

    out_dir = os.path.join(STATS_SAVE_DIR, '_phase06_summary')
    os.makedirs(out_dir, exist_ok=True)
    regime_abl_grid.to_csv(os.path.join(out_dir, 'regime_ablation_full_grid.csv'), index=False)
    regime_abl_summary.to_csv(os.path.join(out_dir, 'regime_ablation_summary.csv'), index=False)
    print(f'\n💾 Saved full grid + summary -> {out_dir}')
else:
    print('  ⚠️ Nothing found yet -- run Phase05 L2/L3 for at least one symbol/horizon first.')


## 9 · Feature-Count Sensitivity — 20 / 40 / 60 / 80 / 100 Features (new)

The one check in this notebook that cannot come from Phase 05's saved artifacts: for the current
`SYMBOL`/`HORIZON` (Section 2), and for every layer, re-runs `select_features_train_only()`
(Section 3.1, identical pipeline to Phase 05) at each `top_k` in `FEATURE_COUNT_GRID`, refits the
**same fixed** `build_lightgbm_fixed()` model on TRAIN+VAL for each feature count, and scores once
on TEST. A flattening curve (TEST metric roughly constant past some `top_k`) says the modelling
pipeline is not sensitive to the exact `N_FEATURES=100` choice Phase 05 uses; a still-rising curve
says more features may still help, or that 100 is close to a genuine peak.

Saved to `Artifacts/stats/{symbol}/{layer}/h{horizon}d/feature_count_sensitivity.csv` — run this
notebook once per `(SYMBOL, HORIZON)` you care about, then the aggregate cell right after this one
builds the full grid.


In [ ]:
def run_feature_count_sensitivity(symbol, horizon, layer, feature_grid=FEATURE_COUNT_GRID):
    task = LAYER_TASK[layer]
    target_col = LAYER_TARGET_COL[layer]
    raw_splits = load_raw_splits(symbol)
    train_df, val_df, test_df = raw_splits.get('train'), raw_splits.get('val'), raw_splits.get('test')
    if train_df is None or val_df is None or test_df is None or target_col not in train_df.columns:
        print(f'  ⚠️ {layer}: missing split(s) or target column -- skipping.')
        return pd.DataFrame()

    candidates = get_candidate_features(train_df)
    rows = []
    for top_k in feature_grid:
        feats, _ = select_features_train_only(train_df, candidates, target_col, task=task, top_k=top_k, verbose=False)
        if not feats:
            print(f'  ⚠️ {layer} top_k={top_k}: 0 features selected -- skipping this arm.')
            continue

        d_train = clean_xy(train_df, feats, target_col)
        d_val   = clean_xy(val_df, feats, target_col)
        d_test  = clean_xy(test_df, feats, target_col)
        X_tv = pd.concat([d_train[feats], d_val[feats]]).sort_index()
        y_tv = pd.concat([d_train[target_col], d_val[target_col]]).sort_index()
        X_test, y_test = d_test[feats], d_test[target_col]

        model = build_lightgbm_fixed(task)
        if task == 'classification':
            model.fit(X_tv, y_tv.map(ENCODE_MAP))
            pred = pd.Series(model.predict(X_test), index=X_test.index).map(DECODE_MAP).values
            proba = model.predict_proba(X_test)
            metrics = evaluate(task, y_test, pred, y_proba=proba, n_features=len(feats))
        else:
            model.fit(X_tv, y_tv)
            pred = model.predict(X_test)
            metrics = evaluate(task, y_test, pred, y_ref=float(y_tv.median()), n_features=len(feats))

        rows.append({'symbol': symbol, 'layer': layer, 'horizon': horizon,
                      'model': ROBUSTNESS_MODEL_NAME, 'n_features_requested': top_k,
                      'n_features_actual': len(feats), **metrics})
        print(f'  {layer} | top_k={top_k:>3d} -> actual={len(feats):>3d} features | '
              + ', '.join(f'{k}={v:.4f}' for k, v in metrics.items() if isinstance(v, float) and not np.isnan(v)))

    df = pd.DataFrame(rows)
    if not df.empty:
        stats_dir = os.path.join(STATS_SAVE_DIR, symbol, layer, f'h{horizon}d')
        os.makedirs(stats_dir, exist_ok=True)
        out_path = os.path.join(stats_dir, 'feature_count_sensitivity.csv')
        df.to_csv(out_path, index=False)
        print(f'  💾 Saved -> {out_path}')
    return df


feature_count_results = {}
for _layer in ALL_LAYERS:
    print(f'\n{"=" * 90}\nFeature-count sensitivity | {SYMBOL} | h={HORIZON}d | {_layer}\n{"=" * 90}')
    feature_count_results[_layer] = run_feature_count_sensitivity(SYMBOL, HORIZON, _layer)

feature_count_this_run = pd.concat([df for df in feature_count_results.values() if not df.empty], ignore_index=True) \
    if any(not df.empty for df in feature_count_results.values()) else pd.DataFrame()
display(feature_count_this_run)


### 9.1 · Aggregate Feature-Count Sensitivity across ALL symbols/horizons already run


In [ ]:
feature_count_grid_all = load_artifact_grid('stats', 'feature_count_sensitivity.csv', verbose=True)
print(f'\nTotal feature-count sensitivity rows across the full grid: {len(feature_count_grid_all)}')
if not feature_count_grid_all.empty:
    metric_col = feature_count_grid_all.apply(
        lambda r: 'f1_macro' if r['layer'] in CLASSIFICATION_LAYERS else 'rmse', axis=1)
    feature_count_grid_all['primary_metric_name'] = metric_col
    feature_count_grid_all['primary_metric_value'] = feature_count_grid_all.apply(
        lambda r: r.get(r['primary_metric_name'], np.nan), axis=1)

    plateau_summary = (
        feature_count_grid_all.groupby(['layer', 'n_features_requested'])['primary_metric_value']
        .mean().reset_index().sort_values(['layer', 'n_features_requested'])
    )
    print('\nMean primary TEST metric (f1_macro for classification / rmse for regression) by feature count:')
    display(plateau_summary)

    out_dir = os.path.join(STATS_SAVE_DIR, '_phase06_summary')
    os.makedirs(out_dir, exist_ok=True)
    feature_count_grid_all.to_csv(os.path.join(out_dir, 'feature_count_sensitivity_full_grid.csv'), index=False)
    plateau_summary.to_csv(os.path.join(out_dir, 'feature_count_sensitivity_summary.csv'), index=False)
    print(f'\n💾 Saved -> {out_dir}')


## 10 · Regime-Threshold Sensitivity — Aggregated

Pure aggregation: Phase 05 L1 (Sections 13.3 / 13B.3) already computes and saves
`regime_threshold_sensitivity.csv` for the ternary regime layer (`L1_1_TRI`) every time it runs.
No new computation here — just the cross-symbol/cross-horizon view of how stable the
Bull/Sideways/Bear boundary is.


In [ ]:
regime_threshold_grid = load_artifact_grid('stats', 'regime_threshold_sensitivity.csv',
                                            layers=['L1_1_TRI'], verbose=True)
print(f'\nTotal regime-threshold sensitivity rows loaded: {len(regime_threshold_grid)}')
if not regime_threshold_grid.empty:
    display(regime_threshold_grid)
    threshold_summary = (
        regime_threshold_grid.groupby('threshold_pct')['agreement_with_target_tri']
        .agg(['mean', 'std', 'min', 'max', 'count']).reset_index()
    )
    print('\nAgreement with the actual target_tri label, averaged across ALL symbols/horizons already run:')
    display(threshold_summary)

    out_dir = os.path.join(STATS_SAVE_DIR, '_phase06_summary')
    os.makedirs(out_dir, exist_ok=True)
    regime_threshold_grid.to_csv(os.path.join(out_dir, 'regime_threshold_sensitivity_full_grid.csv'), index=False)
    threshold_summary.to_csv(os.path.join(out_dir, 'regime_threshold_sensitivity_summary.csv'), index=False)
    print(f'\n💾 Saved -> {out_dir}')
else:
    print('  ⚠️ Nothing found yet -- run Phase05 L1 for at least one symbol/horizon first.')


## 11 · Walk-Forward Stability — new

For the current `SYMBOL`/`HORIZON`, and for every layer, refits the **same fixed**
`build_lightgbm_fixed()` model across `WF_SPLITS` `PurgedWalkForwardCV` folds carved out of
TRAIN+VAL (frozen Phase 05 selected features — Section 3), and reports the mean and
coefficient-of-variation (`std/mean`) of the primary metric across folds. A high CV flags a model
whose performance depends heavily on which slice of history it happened to validate on — a
genuine robustness concern the review's "Selection reproducibility" and "Walk-forward Stability"
asks are both getting at.


In [ ]:
def run_walk_forward_stability(symbol, horizon, layer):
    task = LAYER_TASK[layer]
    res = build_xy_from_frozen(symbol, layer, horizon, 'train')
    res_val = build_xy_from_frozen(symbol, layer, horizon, 'val')
    if res is None or res_val is None:
        print(f'  ⚠️ {layer}: no frozen selection / data available for {symbol} h={horizon}d -- '
              f'run Phase 05 for this combination first. Skipping.')
        return pd.DataFrame()

    X_tr, y_tr, _ = res
    X_va, y_va, _ = res_val
    X = pd.concat([X_tr, X_va]).sort_index()
    y = pd.concat([y_tr, y_va]).sort_index()

    rows = []
    cv = get_cv_splitter(WF_SPLITS)
    for fold_i, (train_idx, val_idx) in enumerate(cv.split(X)):
        X_fold_tr, y_fold_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_fold_va, y_fold_va = X.iloc[val_idx], y.iloc[val_idx]
        model = build_lightgbm_fixed(task)
        if task == 'classification':
            model.fit(X_fold_tr, y_fold_tr.map(ENCODE_MAP))
            pred = pd.Series(model.predict(X_fold_va), index=X_fold_va.index).map(DECODE_MAP).values
            proba = model.predict_proba(X_fold_va)
            metrics = evaluate(task, y_fold_va, pred, y_proba=proba)
        else:
            model.fit(X_fold_tr, y_fold_tr)
            pred = model.predict(X_fold_va)
            metrics = evaluate(task, y_fold_va, pred, y_ref=float(y_fold_tr.median()))
        rows.append({'symbol': symbol, 'layer': layer, 'horizon': horizon, 'fold': fold_i,
                      'n_train': len(train_idx), 'n_val': len(val_idx),
                      'model': ROBUSTNESS_MODEL_NAME, **metrics})

    df = pd.DataFrame(rows)
    if df.empty:
        print(f'  ⚠️ {layer}: PurgedWalkForwardCV produced 0 usable folds (too little history for '
              f'WF_SPLITS={WF_SPLITS} + purge/embargo={PURGE_DAYS}d/{EMBARGO_DAYS}d) -- skipping.')
        return df

    metric_name = 'f1_macro' if task == 'classification' else 'rmse'
    mean_v, std_v = df[metric_name].mean(), df[metric_name].std()
    cv_pct = (std_v / abs(mean_v) * 100) if mean_v else np.nan
    print(f'  {layer}: {metric_name} across {len(df)} folds -> mean={mean_v:.4f}  std={std_v:.4f}  CV={cv_pct:.1f}%')

    stats_dir = os.path.join(STATS_SAVE_DIR, symbol, layer, f'h{horizon}d')
    os.makedirs(stats_dir, exist_ok=True)
    out_path = os.path.join(stats_dir, 'walk_forward_stability.csv')
    df.to_csv(out_path, index=False)
    print(f'  💾 Saved -> {out_path}')
    return df


wf_stability_results = {}
for _layer in ALL_LAYERS:
    print(f'\n{"=" * 90}\nWalk-forward stability | {SYMBOL} | h={HORIZON}d | {_layer}\n{"=" * 90}')
    wf_stability_results[_layer] = run_walk_forward_stability(SYMBOL, HORIZON, _layer)


### 11.1 · Aggregate Walk-Forward Stability across ALL symbols/horizons already run


In [ ]:
wf_stability_grid = load_artifact_grid('stats', 'walk_forward_stability.csv', verbose=True)
print(f'\nTotal walk-forward-stability rows across the full grid: {len(wf_stability_grid)}')
if not wf_stability_grid.empty:
    wf_stability_grid['primary_metric_name'] = wf_stability_grid['layer'].map(
        lambda l: 'f1_macro' if l in CLASSIFICATION_LAYERS else 'rmse')
    wf_stability_grid['primary_metric_value'] = wf_stability_grid.apply(
        lambda r: r.get(r['primary_metric_name'], np.nan), axis=1)

    stability_summary = (
        wf_stability_grid.groupby(['symbol', 'layer', 'horizon'])['primary_metric_value']
        .agg(['mean', 'std', 'count']).reset_index()
    )
    stability_summary['cv_pct'] = (stability_summary['std'] / stability_summary['mean'].abs() * 100)
    stability_summary = stability_summary.sort_values('cv_pct', ascending=False)
    print('\nFold-to-fold coefficient of variation, highest (least stable) first:')
    display(stability_summary)

    out_dir = os.path.join(STATS_SAVE_DIR, '_phase06_summary')
    os.makedirs(out_dir, exist_ok=True)
    wf_stability_grid.to_csv(os.path.join(out_dir, 'walk_forward_stability_full_grid.csv'), index=False)
    stability_summary.to_csv(os.path.join(out_dir, 'walk_forward_stability_summary.csv'), index=False)
    print(f'\n💾 Saved -> {out_dir}')


## 12 · Predictive-Superiority Tests — Aggregated (McNemar / Diebold-Mariano / Bootstrap CI)

Pure aggregation of Phase 05's own significance tests: `bootstrap_ci.csv` + `mcnemar_pairwise.csv`
for the two classification layers (L1), `diebold_mariano.csv` for the five regression layers
(L2/L3), across the full symbol × horizon grid.


In [ ]:
bootstrap_grid = load_artifact_grid('stats', 'bootstrap_ci.csv', layers=CLASSIFICATION_LAYERS, verbose=True)
mcnemar_grid   = load_artifact_grid('stats', 'mcnemar_pairwise.csv', layers=CLASSIFICATION_LAYERS, verbose=True)
dm_grid        = load_artifact_grid('stats', 'diebold_mariano.csv', layers=REGRESSION_LAYERS, verbose=True)

print(f'\nBootstrap CI rows: {len(bootstrap_grid)}  |  McNemar rows: {len(mcnemar_grid)}  |  '
      f'Diebold-Mariano rows: {len(dm_grid)}')

if not mcnemar_grid.empty:
    mcnemar_sig_rate = (
        mcnemar_grid.groupby(['layer'])['significant_at_0.05']
        .mean().rename('pct_pairs_significant').reset_index()
    )
    print('\nMcNemar: fraction of model-pairs significantly different at 5%, by layer:')
    display(mcnemar_sig_rate)

if not dm_grid.empty:
    dm_sig_rate = (
        dm_grid.assign(significant=lambda d: d['p_value'] < 0.05)
        .groupby(['layer'])['significant'].mean().rename('pct_pairs_significant').reset_index()
    )
    print('\nDiebold-Mariano: fraction of model-pairs significantly different at 5%, by layer:')
    display(dm_sig_rate)

out_dir = os.path.join(STATS_SAVE_DIR, '_phase06_summary')
os.makedirs(out_dir, exist_ok=True)
for name, df in [('bootstrap_ci_full_grid', bootstrap_grid), ('mcnemar_pairwise_full_grid', mcnemar_grid),
                  ('diebold_mariano_full_grid', dm_grid)]:
    if not df.empty:
        df.to_csv(os.path.join(out_dir, f'{name}.csv'), index=False)
print(f'\n💾 Saved full grids -> {out_dir}')


## 13 · Regime-Conditional Performance — Aggregated + a New Cross-Layer (L1 → L2/L3) Join

Three pieces:

1. **Aggregation**: every `conditional_performance.csv` (L2's Low/High-value-bucket check,
   Section 11.24/11B.24/11C.24) Phase 05 has already saved, across the grid.
2. **Aggregation**: every `high_regime_bias_check.csv` (L2's high-value-regime systematic
   underprediction flag, same section) Phase 05 has already saved.
3. **New cross-layer join**: for each (symbol, horizon) where BOTH `L1_1_TRI` and an L2/L3 layer
   have saved TEST predictions, joins `L1_1_TRI`'s predicted regime (Bull/Sideways/Bear, from its
   best model's `predictions/.../*_test_predictions.csv`) onto that L2/L3 layer's own TEST
   predictions by `row_index` (the shared date index), then reports each L2/L3 model's RMSE/MAE
   **conditional on the ACTUAL MARKET REGIME** rather than L2's own within-layer proxy (a bucket of
   its own target's high/low values). This is the more literal reading of the review's "Low-vol
   regime vs High-vol regime" / H5 ask -- regime as L1 defines it, not as a stand-in from L2's own
   target distribution.


In [ ]:
# ── 1) Aggregation of L2's own conditional_performance.csv + high_regime_bias_check.csv ────
conditional_perf_grid = load_artifact_grid('stats', 'conditional_performance.csv',
                                            layers=['L1_2_RVOL', 'L1_2_MAXRET', 'L1_2_MAXDRAW'], verbose=True)
high_regime_bias_grid = load_artifact_grid('stats', 'high_regime_bias_check.csv',
                                            layers=['L1_2_RVOL', 'L1_2_MAXRET', 'L1_2_MAXDRAW'], verbose=True)
print(f'Total conditional-performance rows loaded: {len(conditional_perf_grid)}')
print(f'Total high-regime bias-check rows loaded: {len(high_regime_bias_grid)}')
if not conditional_perf_grid.empty:
    display(conditional_perf_grid.groupby(['layer', 'regime_bucket'])[['rmse', 'mae']].mean().reset_index())
if not high_regime_bias_grid.empty:
    print('\nFraction of (symbol, horizon, model) combinations showing systematic '
          'underprediction in the high-value regime, by layer:')
    display(high_regime_bias_grid.groupby('layer')['systematic_underprediction_high_regime']
            .mean().rename('pct_systematic_underprediction').reset_index())

# ── 2) New cross-layer join: L1's predicted regime -> L2/L3's own TEST predictions ──────────
def find_best_model_predictions(symbol, layer, horizon):
    """Picks the best-leaderboard TEST-split model for (symbol, layer, horizon) and loads its
    saved per-row predictions CSV. Returns a DataFrame indexed by row_index, or None."""
    lb_path = os.path.join(RESULTS_SAVE_DIR, symbol, layer, f'h{horizon}d', 'leaderboard.csv')
    if not os.path.exists(lb_path):
        return None
    lb = pd.read_csv(lb_path)
    test_rows = lb[lb['split'] == 'test']
    if test_rows.empty:
        return None
    metric = 'f1_macro' if layer in CLASSIFICATION_LAYERS else 'rmse'
    ascending = metric == 'rmse'
    best_model = test_rows.sort_values(metric, ascending=ascending).iloc[0]['model']
    pred_path = os.path.join(PRED_SAVE_DIR, symbol, layer, f'h{horizon}d', f'{best_model}_test_predictions.csv')
    if not os.path.exists(pred_path):
        return None
    df = pd.read_csv(pred_path)
    df['best_model'] = best_model
    return df.set_index('row_index')


cross_layer_rows = []
for symbol in ALL_SYMBOLS:
    for horizon in ALL_HORIZONS_FOR_STATS:
        regime_pred = find_best_model_predictions(symbol, 'L1_1_TRI', horizon)
        if regime_pred is None:
            continue
        regime_series = regime_pred['y_pred'].map(REGIME_LABEL_MAP)

        for layer in ['L1_2_RVOL', 'L1_2_MAXRET', 'L1_2_MAXDRAW', 'L1_3_RET', 'L1_3_PCT']:
            target_pred = find_best_model_predictions(symbol, layer, horizon)
            if target_pred is None:
                continue
            joined = target_pred.join(regime_series.rename('market_regime'), how='inner')
            if joined.empty:
                continue
            for regime_name, g in joined.groupby('market_regime'):
                if len(g) < 3:
                    continue
                cross_layer_rows.append({
                    'symbol': symbol, 'horizon': horizon, 'layer': layer,
                    'model': g['best_model'].iloc[0], 'market_regime': regime_name, 'n': len(g),
                    'rmse': float(np.sqrt(np.mean(g['error'] ** 2))),
                    'mae': float(np.mean(np.abs(g['error']))),
                })

cross_layer_regime_df = pd.DataFrame(cross_layer_rows)
print(f'\nCross-layer (L1-regime-conditional) rows built: {len(cross_layer_regime_df)}')
if not cross_layer_regime_df.empty:
    display(cross_layer_regime_df.sort_values(['layer', 'symbol', 'horizon', 'market_regime']))

out_dir = os.path.join(STATS_SAVE_DIR, '_phase06_summary')
os.makedirs(out_dir, exist_ok=True)
if not conditional_perf_grid.empty:
    conditional_perf_grid.to_csv(os.path.join(out_dir, 'conditional_performance_full_grid.csv'), index=False)
if not cross_layer_regime_df.empty:
    cross_layer_regime_df.to_csv(os.path.join(out_dir, 'cross_layer_market_regime_conditional.csv'), index=False)
print(f'\n💾 Saved -> {out_dir}')


## 14 · Horizon-Conditional Performance — Aggregated

Aggregates every layer's `leaderboard.csv` (TEST split only) across `h ∈ {1,3,5,10,21}` for the
current `SYMBOL` (change Section 2's `SYMBOL` and rerun to see another index), showing how each
model's primary metric moves with forecast horizon -- the H4/H5-adjacent "does performance /
information usefulness change with horizon" question, read directly off the leaderboards Phase 05
already produced rather than a fresh experiment.


In [ ]:
leaderboard_grid = load_artifact_grid('results', 'leaderboard.csv', symbols=[SYMBOL], verbose=True)
leaderboard_test = leaderboard_grid[leaderboard_grid['split'] == 'test'] if not leaderboard_grid.empty else leaderboard_grid
print(f'\n{SYMBOL}: {len(leaderboard_test)} TEST-split leaderboard rows across all horizons already run.')

if not leaderboard_test.empty:
    for layer in ALL_LAYERS:
        sub = leaderboard_test[leaderboard_test['layer'] == layer]
        if sub.empty:
            continue
        metric = 'f1_macro' if layer in CLASSIFICATION_LAYERS else 'rmse'
        pivot = sub.pivot_table(index='model', columns='horizon', values=metric)
        print(f'\n{layer} — {metric} by horizon (rows=model, columns=horizon):')
        display(pivot)

    out_dir = os.path.join(STATS_SAVE_DIR, '_phase06_summary')
    os.makedirs(out_dir, exist_ok=True)
    leaderboard_test.to_csv(os.path.join(out_dir, f'horizon_conditional_leaderboard_{SYMBOL}.csv'), index=False)
    print(f'\n💾 Saved -> {out_dir}')


## 15 · Cross-Index Heterogeneity — Aggregated

The other half of the same `leaderboard.csv` grid, sliced the other way: for a fixed `HORIZON`
(Section 2), compares each layer/model's TEST metric across `NIFTY50` / `BANKNIFTY` /
`NIFTYNEXT50`. Large spreads across symbols flag a finding that only holds for one index, not a
genuinely general one -- a legitimate robustness question the review's own architecture calls out
as its own numbered section.


In [ ]:
leaderboard_grid_h = load_artifact_grid('results', 'leaderboard.csv', horizons=[HORIZON], verbose=True)
leaderboard_test_h = leaderboard_grid_h[leaderboard_grid_h['split'] == 'test'] if not leaderboard_grid_h.empty else leaderboard_grid_h
print(f'\nh={HORIZON}d: {len(leaderboard_test_h)} TEST-split leaderboard rows across all symbols already run.')

cross_index_summary_frames = []
if not leaderboard_test_h.empty:
    for layer in ALL_LAYERS:
        sub = leaderboard_test_h[leaderboard_test_h['layer'] == layer]
        if sub.empty:
            continue
        metric = 'f1_macro' if layer in CLASSIFICATION_LAYERS else 'rmse'
        pivot = sub.pivot_table(index='model', columns='symbol', values=metric)
        pivot['spread'] = pivot.max(axis=1) - pivot.min(axis=1)
        print(f'\n{layer} — {metric} by symbol (h={HORIZON}d), with the cross-symbol spread:')
        display(pivot.sort_values('spread', ascending=False))
        cross_index_summary_frames.append(pivot.reset_index().assign(layer=layer, metric=metric))

    out_dir = os.path.join(STATS_SAVE_DIR, '_phase06_summary')
    os.makedirs(out_dir, exist_ok=True)
    leaderboard_test_h.to_csv(os.path.join(out_dir, f'cross_index_leaderboard_h{HORIZON}d.csv'), index=False)
    print(f'\n💾 Saved -> {out_dir}')


## 16 · Economic Significance — Aggregated

Pure aggregation of Phase 05 L3's `economic_value.csv` (Section 15 there) across the full
symbol × horizon grid — Sharpe ratio, max drawdown, and transaction-cost-adjusted return per model,
compared against each symbol/horizon's own Buy & Hold reference.


In [ ]:
econ_value_grid = load_artifact_grid('stats', 'economic_value.csv', layers=['L1_3_RET', 'L1_3_PCT'], verbose=True)
print(f'\nTotal economic-value rows loaded: {len(econ_value_grid)}')
if not econ_value_grid.empty:
    display(econ_value_grid.sort_values(['symbol', 'horizon', 'sharpe_ratio'], ascending=[True, True, False]))

    beat_bh_rows = []
    for (symbol, layer, horizon), g in econ_value_grid.groupby(['symbol', 'layer', 'horizon']):
        bh = g[g['model'] == 'Buy_and_Hold']
        if bh.empty:
            continue
        bh_sharpe = bh.iloc[0]['sharpe_ratio']
        n_beat = int((g[g['model'] != 'Buy_and_Hold']['sharpe_ratio'] > bh_sharpe).sum())
        n_total = int((g['model'] != 'Buy_and_Hold').sum())
        beat_bh_rows.append({'symbol': symbol, 'layer': layer, 'horizon': horizon,
                              'buy_and_hold_sharpe': bh_sharpe, 'n_models_beating_bh_sharpe': n_beat,
                              'n_models_total': n_total})
    beat_bh_df = pd.DataFrame(beat_bh_rows)
    print('\nHow many models beat Buy & Hold on Sharpe, per (symbol, layer, horizon):')
    display(beat_bh_df)

    out_dir = os.path.join(STATS_SAVE_DIR, '_phase06_summary')
    os.makedirs(out_dir, exist_ok=True)
    econ_value_grid.to_csv(os.path.join(out_dir, 'economic_value_full_grid.csv'), index=False)
    beat_bh_df.to_csv(os.path.join(out_dir, 'economic_value_vs_buy_and_hold.csv'), index=False)
    print(f'\n💾 Saved -> {out_dir}')
else:
    print('  ⚠️ Nothing found yet -- run Phase05 L3 for at least one symbol/horizon first.')


## 17 · Robustness Summary Tables — One Consolidated Scorecard

Pulls together a single pass/fail-style scorecard from every section above, per layer: whether H2
(information-set ablation), H3 (regime ablation), and the economic-value check show a
statistically/economically meaningful effect anywhere in the grid already run, plus the
feature-count plateau point and the fold-to-fold stability CV. This is the table meant to go
straight into the report's robustness/ablation appendix.


In [ ]:
scorecard_rows = []
for layer in ALL_LAYERS:
    row = {'layer': layer, 'task': LAYER_TASK[layer]}

    if not h1_grid.empty:
        g = h1_grid[h1_grid['layer'] == layer]
        row['h1_pct_significant_vs_benchmarks'] = float(g['significant_at_0.05'].fillna(False).astype(bool).mean()) if not g.empty else np.nan
    else:
        row['h1_pct_significant_vs_benchmarks'] = np.nan

    if not info_set_grid.empty:
        g = info_set_grid[info_set_grid['layer'] == layer]
        row['h2_mean_delta_rmse_market_to_full'] = float(g.groupby('symbol')['delta_rmse_vs_prev_arm'].sum().mean()) if not g.empty else np.nan
    else:
        row['h2_mean_delta_rmse_market_to_full'] = np.nan

    if not regime_abl_grid.empty:
        g = regime_abl_grid[regime_abl_grid['layer'] == layer]
        row['h3_pct_significant_regime_helps'] = float((g['dm_significant_at_0.05'].fillna(False) & (g['delta_rmse'] > 0)).mean()) if not g.empty else np.nan
    else:
        row['h3_pct_significant_regime_helps'] = np.nan

    if not feature_count_grid_all.empty:
        g = feature_count_grid_all[feature_count_grid_all['layer'] == layer]
        if not g.empty:
            best_k = g.groupby('n_features_requested')['primary_metric_value'].mean()
            row['feature_count_best_k'] = int(best_k.idxmin() if layer in REGRESSION_LAYERS else best_k.idxmax())
        else:
            row['feature_count_best_k'] = np.nan
    else:
        row['feature_count_best_k'] = np.nan

    if not wf_stability_grid.empty:
        g = wf_stability_grid[wf_stability_grid['layer'] == layer]
        if not g.empty:
            m, s = g['primary_metric_value'].mean(), g['primary_metric_value'].std()
            row['walk_forward_cv_pct'] = float(s / abs(m) * 100) if m else np.nan
        else:
            row['walk_forward_cv_pct'] = np.nan
    else:
        row['walk_forward_cv_pct'] = np.nan

    if layer in ('L1_3_RET', 'L1_3_PCT') and not econ_value_grid.empty:
        g = econ_value_grid[(econ_value_grid['layer'] == layer) & (econ_value_grid['model'] != 'Buy_and_Hold')]
        row['pct_beats_buy_and_hold_sharpe'] = float((g['sharpe_ratio'] > 0).mean()) if not g.empty else np.nan
    else:
        row['pct_beats_buy_and_hold_sharpe'] = np.nan

    scorecard_rows.append(row)

robustness_scorecard = pd.DataFrame(scorecard_rows)
print('Robustness Scorecard — one row per layer:')
display(robustness_scorecard)

out_dir = os.path.join(STATS_SAVE_DIR, '_phase06_summary')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'robustness_scorecard.csv')
robustness_scorecard.to_csv(out_path, index=False)
print(f'\n💾 Saved final robustness scorecard -> {out_path}')
